# Fund Data Chatbot (Holdings & Trades)

This notebook builds a simple **data-grounded chatbot** that answers questions **only** from the provided CSV files (`holdings.csv`, `trades.csv`).

**Rule enforced:** If an answer cannot be derived from these files, the bot responds:

> *“Sorry can not find the answer”*

## 1. Load Data

In [14]:
import pandas as pd

# Load CSV files
holdings = pd.read_csv("holdings.csv")
trades = pd.read_csv("trades.csv")

print("Holdings shape:", holdings.shape)
print("Trades shape:", trades.shape)

display(holdings.head())
display(trades.head())





Holdings shape: (1022, 25)
Trades shape: (649, 31)


,AsOfDate,OpenDate,CloseDate,ShortName,PortfolioName,StrategyRefShortName,Strategy1RefShortName,Strategy2RefShortName,CustodianName,DirectionName,...,StartPrice,Price,StartFXRate,FXRate,MV_Local,MV_Base,PL_DTD,PL_QTD,PL_MTD,PL_YTD
0,01/08/23,04/03/20,NaN,Garfield,Garfield,Default,Asset,DefaultS2,Well Prime,Long,...,96.0,96.0,1.33,1.33,568320.00,7.558656e+05,92.5040,10833.7294,92.5040,41054.5854
1,01/08/23,04/03/20,NaN,Garfield,Garfield,Default,Asset,DefaultS2,Well Prime,Long,...,96.0,96.0,1.33,1.33,84.48,1.123584e+02,0.0138,1.6104,0.0138,6.1027
2,01/08/23,04/03/20,NaN,Garfield,Garfield,Default,Asset,DefaultS2,Well Prime,Long,...,96.0,96.0,1.33,1.33,756000.00,1.005480e+06,123.0523,14411.4221,123.0523,54612.3074
3,01/08/23,04/03/20,NaN,Garfield,Garfield,Default,Asset,DefaultS2,Well Prime,Long,...,96.0,96.0,1.33,1.33,484800.00,6.447840e+05,78.9097,9241.6104,78.9097,35021.2257
4,01/08/23,04/03/20,NaN,Heather,Heather,Default,Asset,DefaultS2,Well Prime,Long,...,96.0,96.0,1.33,1.33,487680.00,6.486144e+05,79.3785,9296.5110,79.3785,35229.2726


,id,RevisionId,AllocationId,TradeTypeName,SecurityId,SecurityType,Name,Ticker,CUSIP,ISIN,...,AllocationFees,AllocationCash,PortfolioName,CustodianName,StrategyName,Strategy1Name,Strategy2Name,Counterparty,AllocationRule,IsCustomAllocation
0,3489863,2,3460886,Buy,270471,Equity,Berry Brand 4/11 Equity,NaN,NaN,NaN,...,2800.00,7.002800e+06,HoldCo 1,JP MORGAN SECURITIES LLC,Default,DefaultS1,DefaultS2,ABGS,Single Fund Rule - HoldCo 1,1
1,3489864,1,3460887,Sell,270471,Equity,Berry Brand 4/11 Equity,NaN,NaN,NaN,...,128.80,6.999871e+06,HoldCo 1,JP MORGAN SECURITIES LLC,Default,DefaultS1,DefaultS2,ABGS,Single Fund Rule - HoldCo 1,0
2,3496826,1,3462756,Sell,290063,Equity,META-US,META,30303M102,US30303M1027,...,46985.99,2.553540e+09,HoldCo 3,CITIGROUP GLOBAL MARKETS INC.,Default,DefaultS1,DefaultS2,ABGS,Single Fund Rule - HoldCo 3,0
3,3496828,3,3462769,Buy,290067,Equity,SPOT-US,SPOT,NaN,LU1778762911,...,20.20,1.098249e+06,HoldCo 11,Goldman Sachs International,Default,Asset,DefaultS2,ABGS,Single Fund Rule - HoldCo 11,1
4,3496829,4,3462770,Buy,290067,Equity,SPOT-US,SPOT,NaN,LU1778762911,...,60.62,3.294749e+06,HoldCo 11,Goldman Sachs International,Default,Asset,DefaultS2,ABGS,Single Fund Rule - HoldCo 11,1


## 2. Helper Functions
We define deterministic functions to answer supported question types.
No internet or external knowledge is used.

In [6]:

def total_holdings_by_fund(fund_name):
    df = holdings[holdings['PortfolioName'].str.contains(fund_name, case=False, na=False)]
    if df.empty:
        return None
    return len(df)

def total_trades_by_fund(fund_name):
    df = trades[trades['PortfolioName'].str.contains(fund_name, case=False, na=False)]
    if df.empty:
        return None
    return len(df)

def yearly_pnl_by_fund():
    if 'PL_YTD' not in holdings.columns:
        return None
    return (
        holdings.groupby('PortfolioName')['PL_YTD']
        .sum()
        .sort_values(ascending=False)
    )


## 3. Chatbot Logic
A lightweight rule-based NLP router maps questions to data functions.
If no rule matches → safe fallback response.

In [7]:

def chatbot(question):
    q = question.lower()
    
    if "total number of holdings" in q:
        fund = q.split("for")[-1].strip()
        ans = total_holdings_by_fund(fund)
        return ans if ans is not None else "Sorry can not find the answer"
    
    if "total number of trades" in q:
        fund = q.split("for")[-1].strip()
        ans = total_trades_by_fund(fund)
        return ans if ans is not None else "Sorry can not find the answer"
    
    if "performed better" in q or "profit and loss" in q:
        pnl = yearly_pnl_by_fund()
        return pnl if pnl is not None else "Sorry can not find the answer"
    
    return "Sorry can not find the answer"


## 4. Example Questions & Output

In [17]:

print(chatbot("Total number of holdings for Garfield"))
print(chatbot("Total number of trades for HoldCo 11"))
print(chatbot("Which funds performed better depending on the yearly Profit and Loss"))
print(chatbot("What is the GDP of India?"))  # should fallback
print(chatbot("Total number of holdings for Garfield"))

chatbot("Total number of holdings for Garfield")
print(chatbot("Which funds performed better depending on the the yearly Profit and Loss"))
print(chatbot("Who is the PM of India"))





221
6
PortfolioName
Ytum                       7.229903e+06
NPSMF1                     3.043387e+05
NPSMF2                     2.975902e+05
NPSMF3                     2.073081e+05
CoYold 1                   2.105159e+04
IG Corp                    2.068600e+00
SMA-L1                    -3.287669e+03
SMA-L2                    -3.337672e+03
SMA-L4                    -3.337672e+03
Hi Yield                  -5.124479e+04
Warren Lee IG             -5.707100e+04
Opium Holdings Partners   -8.860095e+06
Platpot                   -3.650601e+07
CoYold 11                 -1.120346e+08
Garfield                  -1.685510e+08
Heather                   -1.815971e+08
Northpoint 401K           -2.186799e+08
MNC Investment Fund       -3.286149e+08
CoYold 7                  -5.000000e+09
Name: PL_YTD, dtype: float64
Sorry can not find the answer
221
PortfolioName
Ytum                       7.229903e+06
NPSMF1                     3.043387e+05
NPSMF2                     2.975902e+05
NPSMF3                 

## 5. Explanation
- The chatbot **never queries the internet**.
- Answers are computed directly from pandas operations.
- Unsupported questions safely return the fixed apology response.

This design guarantees **data fidelity** and prevents hallucinations.

## 6. Video Demo (How to Record)
1. Open this notebook in Jupyter / Colab
2. Run all cells top to bottom
3. Ask live questions by calling `chatbot("your question")`
4. Record screen using **Loom / OBS / Zoom**
5. Explain:
   - Data loading
   - Question routing
   - Example outputs

_Attach the video link when submitting._